# Notebook 03 — Forecasting problem design

Freeze the forecasting problem contract (targets, horizons, leakage-safe availability rules, chronological splits, metrics, and evaluation slices) **before** baselines/modeling.


## 0. Imports and setup


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
# Dependency preflight: parquet support (read/write)
try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "Missing optional dependency 'pyarrow' required for parquet IO in this notebook. "
        "Install with: pip install pyarrow"
    ) from exc


In [3]:
def find_project_root(start_path: Path) -> Path:
    """Find the project root by walking upward from a start path.

    Markers searched (in priority order):
    - pyproject.toml
    - README.md
    - data/ directory

    The notebook must not assume the execution CWD is already the project root.
    """

    best_candidate = None
    best_score = -1

    for candidate in [start_path, *start_path.parents]:
        has_pyproject = (candidate / 'pyproject.toml').exists()
        has_readme = (candidate / 'README.md').exists()
        has_data_dir = (candidate / 'data').is_dir()

        score = 0
        score += 4 if has_pyproject else 0
        score += 2 if has_readme else 0
        score += 1 if has_data_dir else 0

        if score > best_score:
            best_candidate = candidate
            best_score = score

        if has_data_dir and (has_pyproject or has_readme):
            return candidate

    if best_candidate is not None and (best_candidate / 'data').is_dir():
        return best_candidate

    raise FileNotFoundError(
        "Could not locate PROJECT_ROOT by walking upward from the current working directory. "
        "Expected to find at least a 'data/' directory and ideally 'README.md' or 'pyproject.toml'."
    )


In [4]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())

ASSEMBLED_PATH = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'beijing_multisite_assembled.parquet'
STATION_STATUS_PATH = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'temporal_audit' / 'station_status_policy.parquet'
EDA_IMPLICATIONS_PATH = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'eda_environmental_signals' / 'eda_modeling_implications.parquet'

OUT_DIR = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'forecasting_contract'
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CONTRACT_JSON = OUT_DIR / 'forecasting_problem_contract.json'
OUT_SPLIT_DEF = OUT_DIR / 'split_definition.parquet'
OUT_LABEL_H1 = OUT_DIR / 'label_index_h1.parquet'
OUT_LABEL_H24 = OUT_DIR / 'label_index_h24.parquet'
OUT_SUMMARY = OUT_DIR / 'contract_summary_tables.parquet'

KEYS = ['station', 'timestamp']
TARGET_COL = 'PM2.5'
ELIGIBLE_STATUSES = {'include', 'caution'}
H_PRIMARY = 24
H_SECONDARY = 1
HORIZONS_HOURS = [H_SECONDARY, H_PRIMARY]

TRAIN_PROP = 0.70
VAL_PROP = 0.15
PURGE_GAP_HOURS = 24

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'OUT_DIR: {OUT_DIR}')


PROJECT_ROOT: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals
OUT_DIR: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals\data\interim\beijing_air_quality\forecasting_contract


## 1. Scope and frozen contract decisions (non-modeling)

This notebook freezes:
- station inclusion policy (`include` + `caution`)
- targets and horizons (primary: h24, secondary: h1)
- leakage-safe availability rules (observed-only inputs up to prediction time `t`)
- global chronological train/validation/test split with 24h purge gaps
- label assignment rules (by prediction time `t`, require `t+h` stays inside same split)
- high-pollution evaluation flags based on train-only p95 thresholds

Out of scope: imputation, feature engineering/lags, baselines, models.


## 2. Load inputs (assembled + station policy + EDA implications)


In [5]:
for p in [ASSEMBLED_PATH, STATION_STATUS_PATH, EDA_IMPLICATIONS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required input: {p}')

df = pd.read_parquet(ASSEMBLED_PATH)
station_status_policy = pd.read_parquet(STATION_STATUS_PATH)
eda_implications = pd.read_parquet(EDA_IMPLICATIONS_PATH)

missing_core = [c for c in KEYS + [TARGET_COL] if c not in df.columns]
if missing_core:
    raise ValueError(f'Missing core columns in assembled data: {missing_core}')

if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
    parsed = pd.to_datetime(df['timestamp'], errors='coerce')
    if int(parsed.isna().sum()) > 0:
        raise ValueError('timestamp could not be parsed cleanly (nulls introduced)')
    df = df.copy()
    df['timestamp'] = parsed

{
    'assembled_shape': tuple(df.shape),
    'n_stations': int(df['station'].nunique()),
    'timestamp_min': str(df['timestamp'].min()),
    'timestamp_max': str(df['timestamp'].max()),
    'station_policy_status_counts': station_status_policy['status'].value_counts(dropna=False).to_dict(),
    'eda_implications_shape': tuple(eda_implications.shape),
}


{'assembled_shape': (420768, 20),
 'n_stations': 12,
 'timestamp_min': '2013-03-01 00:00:00',
 'timestamp_max': '2017-02-28 23:00:00',
 'station_policy_status_counts': {'include': 12},
 'eda_implications_shape': (6, 5)}

## 3. Station inclusion policy (include/caution) and key validation


In [6]:
stations_df = set(df['station'].astype(str).unique().tolist())
stations_policy = set(station_status_policy['station'].astype(str).unique().tolist())
if stations_df != stations_policy:
    raise ValueError('Station set mismatch between assembled data and station_status_policy')

eligible_stations = (
    station_status_policy.loc[station_status_policy['status'].isin(ELIGIBLE_STATUSES), 'station']
    .astype(str)
    .tolist()
)
if not eligible_stations:
    raise ValueError('No include/caution stations available for forecasting contract')

df_filt = df[df['station'].astype(str).isin(set(eligible_stations))].copy()

key_nulls = {
    'station_nulls': int(df_filt['station'].isna().sum()),
    'timestamp_nulls': int(df_filt['timestamp'].isna().sum()),
}
if key_nulls['station_nulls'] > 0 or key_nulls['timestamp_nulls'] > 0:
    raise ValueError(f'Null keys found after filtering: {key_nulls}')

{
    'eligible_station_count': int(len(set(eligible_stations))),
    'filtered_shape': tuple(df_filt.shape),
    'key_nulls': key_nulls,
}


{'eligible_station_count': 12,
 'filtered_shape': (420768, 20),
 'key_nulls': {'station_nulls': 0, 'timestamp_nulls': 0}}

## 4. Define horizons and label construction rules

Label definition: for each station and prediction timestamp `t_pred`, define `t_target = t_pred + h` and `y = PM2.5(t_target)`.
A labeled example is eligible only when `t_target` exists and stays inside the **same split** as `t_pred` (no split or purge-gap crossing).


In [7]:
HORIZON_SPECS = [
    {'horizon_hours': H_SECONDARY, 'name': 'h1'},
    {'horizon_hours': H_PRIMARY, 'name': 'h24'},
]
pd.DataFrame(HORIZON_SPECS)


,horizon_hours,name
0,1,h1
1,24,h24


## 5. Define global chronological split windows


In [8]:
min_ts = df_filt['timestamp'].min()
max_ts = df_filt['timestamp'].max()

grid_start = pd.Timestamp(min_ts).floor('h')
grid_end = pd.Timestamp(max_ts).ceil('h')
hourly_grid = pd.date_range(grid_start, grid_end, freq='h')

N = int(len(hourly_grid))
if N < 24 * 30:
    print('Caution: very short hourly grid; split proportions may be unstable')

train_end_idx = int(np.floor(TRAIN_PROP * N)) - 1
val_end_idx = int(np.floor((TRAIN_PROP + VAL_PROP) * N)) - 1
if train_end_idx < 0 or val_end_idx <= train_end_idx or val_end_idx >= (N - 1):
    raise ValueError('Invalid split indices computed from proportions')

base_train_start = hourly_grid[0]
base_train_end = hourly_grid[train_end_idx]
base_val_start = hourly_grid[train_end_idx + 1]
base_val_end = hourly_grid[val_end_idx]
base_test_start = hourly_grid[val_end_idx + 1]
base_test_end = hourly_grid[-1]

pd.DataFrame(
    [
        {'split': 'train', 'start': base_train_start, 'end': base_train_end},
        {'split': 'val', 'start': base_val_start, 'end': base_val_end},
        {'split': 'test', 'start': base_test_start, 'end': base_test_end},
    ]
)


,split,start,end
0,train,2013-03-01 00:00:00,2015-12-18 15:00:00
1,val,2015-12-18 16:00:00,2016-07-24 19:00:00
2,test,2016-07-24 20:00:00,2017-02-28 23:00:00


## 6. Apply purge gaps (24h)

Purge gaps remove prediction timestamps close to split boundaries to reduce boundary leakage risk when later notebooks introduce lagged features.


In [9]:
gap = pd.Timedelta(hours=int(PURGE_GAP_HOURS))

train_purge_start = base_train_end + pd.Timedelta(hours=1)
train_purge_end = base_train_end + gap

val_start = train_purge_end + pd.Timedelta(hours=1)
val_end = base_val_end

val_purge_start = val_end + pd.Timedelta(hours=1)
val_purge_end = val_end + gap

test_start = val_purge_end + pd.Timedelta(hours=1)
test_end = base_test_end

if val_start > val_end:
    raise ValueError('Purge gap after train collapses validation window')
if test_start > test_end:
    raise ValueError('Purge gap after validation collapses test window')

pd.DataFrame(
    [
        {'segment': 'train', 'start': base_train_start, 'end': base_train_end},
        {'segment': 'train_purge_gap', 'start': train_purge_start, 'end': train_purge_end},
        {'segment': 'val', 'start': val_start, 'end': val_end},
        {'segment': 'val_purge_gap', 'start': val_purge_start, 'end': val_purge_end},
        {'segment': 'test', 'start': test_start, 'end': test_end},
    ]
)


,segment,start,end
0,train,2013-03-01 00:00:00,2015-12-18 15:00:00
1,train_purge_gap,2015-12-18 16:00:00,2015-12-19 15:00:00
2,val,2015-12-19 16:00:00,2016-07-24 19:00:00
3,val_purge_gap,2016-07-24 20:00:00,2016-07-25 19:00:00
4,test,2016-07-25 20:00:00,2017-02-28 23:00:00


## 7. Build split_definition table


In [10]:
split_df = pd.DataFrame({'timestamp': hourly_grid})
split_df['split'] = 'out_of_range'
split_df['note'] = ''

def _mark_window(mask, split_name: str, note: str = '') -> None:
    split_df.loc[mask, 'split'] = split_name
    if note:
        split_df.loc[mask, 'note'] = note

ts = split_df['timestamp']
_mark_window((ts >= base_train_start) & (ts <= base_train_end), 'train')
_mark_window((ts >= train_purge_start) & (ts <= train_purge_end), 'purge_gap', 'train_gap_after')
_mark_window((ts >= val_start) & (ts <= val_end), 'val')
_mark_window((ts >= val_purge_start) & (ts <= val_purge_end), 'purge_gap', 'val_gap_after')
_mark_window((ts >= test_start) & (ts <= test_end), 'test')

split_order_map = {'train': 0, 'purge_gap': 1, 'val': 2, 'test': 3, 'out_of_range': 9}
split_df['split_order'] = split_df['split'].map(split_order_map).astype(int)

# Sanity: no overlapping labels by construction; just surface counts
split_counts = split_df['split'].value_counts(dropna=False).rename_axis('split').reset_index(name='n_hours')
split_counts


,split,n_hours
0,train,24544
1,val,5236
2,test,5236
3,purge_gap,48


## 8. Build label index (h1)


In [11]:
def build_label_index(df_in: pd.DataFrame, split_def: pd.DataFrame, horizon_hours: int) -> pd.DataFrame:
    """Create a label index for a single horizon.

    Assignment: by prediction timestamp t_pred.
    Eligibility: split(t_pred) in {train,val,test} AND split(t_target) == split(t_pred).
    Target: y = PM2.5 at t_target, required non-null (no imputation).
    """

    h = int(horizon_hours)

    base = df_in[['station', 'timestamp']].copy()
    base['station'] = base['station'].astype(str)
    base = base.rename(columns={'timestamp': 't_pred'})

    split_map = split_def.set_index('timestamp')['split']
    base['split'] = base['t_pred'].map(split_map)

    base['t_target'] = base['t_pred'] + pd.Timedelta(hours=h)
    base['split_target'] = base['t_target'].map(split_map)

    eligible_splits = {'train', 'val', 'test'}
    keep = base['split'].isin(eligible_splits) & (base['split'] == base['split_target'])

    target_lookup = df_in[['station', 'timestamp', TARGET_COL]].copy()
    target_lookup['station'] = target_lookup['station'].astype(str)
    target_lookup = target_lookup.rename(columns={'timestamp': 't_target', TARGET_COL: 'y'})

    li = base.merge(target_lookup, on=['station', 't_target'], how='left')
    li['horizon_hours'] = h

    li = li.loc[keep].copy()
    li = li.loc[li['y'].notna()].copy()

    li = li[['station', 't_pred', 'horizon_hours', 't_target', 'split', 'y']].sort_values(['station', 't_pred'])
    li = li.reset_index(drop=True)
    return li

label_h1 = build_label_index(df_filt, split_df, horizon_hours=H_SECONDARY)
label_h1.head(10)


,station,t_pred,horizon_hours,t_target,split,y
0,Aotizhongxin,2013-03-01 00:00:00,1,2013-03-01 01:00:00,train,8.0
1,Aotizhongxin,2013-03-01 01:00:00,1,2013-03-01 02:00:00,train,7.0
2,Aotizhongxin,2013-03-01 02:00:00,1,2013-03-01 03:00:00,train,6.0
3,Aotizhongxin,2013-03-01 03:00:00,1,2013-03-01 04:00:00,train,3.0
4,Aotizhongxin,2013-03-01 04:00:00,1,2013-03-01 05:00:00,train,5.0
5,Aotizhongxin,2013-03-01 05:00:00,1,2013-03-01 06:00:00,train,3.0
6,Aotizhongxin,2013-03-01 06:00:00,1,2013-03-01 07:00:00,train,3.0
7,Aotizhongxin,2013-03-01 07:00:00,1,2013-03-01 08:00:00,train,3.0
8,Aotizhongxin,2013-03-01 08:00:00,1,2013-03-01 09:00:00,train,3.0
9,Aotizhongxin,2013-03-01 09:00:00,1,2013-03-01 10:00:00,train,3.0


## 9. Build label index (h24)


In [12]:
label_h24 = build_label_index(df_filt, split_df, horizon_hours=H_PRIMARY)
label_h24.head(10)


,station,t_pred,horizon_hours,t_target,split,y
0,Aotizhongxin,2013-03-01 00:00:00,24,2013-03-02 00:00:00,train,22.0
1,Aotizhongxin,2013-03-01 01:00:00,24,2013-03-02 01:00:00,train,14.0
2,Aotizhongxin,2013-03-01 02:00:00,24,2013-03-02 02:00:00,train,13.0
3,Aotizhongxin,2013-03-01 03:00:00,24,2013-03-02 03:00:00,train,3.0
4,Aotizhongxin,2013-03-01 04:00:00,24,2013-03-02 04:00:00,train,3.0
5,Aotizhongxin,2013-03-01 05:00:00,24,2013-03-02 05:00:00,train,9.0
6,Aotizhongxin,2013-03-01 06:00:00,24,2013-03-02 06:00:00,train,4.0
7,Aotizhongxin,2013-03-01 07:00:00,24,2013-03-02 07:00:00,train,3.0
8,Aotizhongxin,2013-03-01 08:00:00,24,2013-03-02 08:00:00,train,3.0
9,Aotizhongxin,2013-03-01 09:00:00,24,2013-03-02 09:00:00,train,10.0


## 10. Define high-pollution thresholds (train-only) and flags


In [13]:
def add_high_pollution_flags(label_index: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    li = label_index.copy()

    train_y = li.loc[li['split'] == 'train', 'y'].astype(float)
    if train_y.empty:
        raise ValueError('No training labels available to compute train-only p95 threshold')

    global_p95 = float(train_y.quantile(0.95))

    station_p95 = (
        li.loc[li['split'] == 'train']
        .groupby('station')['y']
        .quantile(0.95)
        .astype(float)
    )

    li['global_p95_value'] = global_p95
    li['station_p95_value'] = li['station'].map(station_p95)

    li['is_high_global_p95'] = li['y'].astype(float) >= global_p95
    li['is_high_station_p95'] = li['y'].astype(float) >= li['station_p95_value'].astype(float)

    meta = {
        'global_p95_value': global_p95,
        'n_stations_station_p95': int(station_p95.shape[0]),
    }
    return li, meta

label_h1_flagged, hp_meta_h1 = add_high_pollution_flags(label_h1)
label_h24_flagged, hp_meta_h24 = add_high_pollution_flags(label_h24)

{'h1': hp_meta_h1, 'h24': hp_meta_h24}


{'h1': {'global_p95_value': 239.0, 'n_stations_station_p95': 12},
 'h24': {'global_p95_value': 239.0, 'n_stations_station_p95': 12}}

## 11. Contract validation checks (no-leakage, boundary, counts)


In [14]:
def summarize_label_index(li: pd.DataFrame) -> pd.DataFrame:
    out = (
        li.groupby(['horizon_hours', 'station', 'split'], as_index=False)
        .agg(
            n_examples=('y', 'size'),
            y_missing_rate=('y', lambda s: float(pd.Series(s).isna().mean())),
            high_global_rate=('is_high_global_p95', lambda s: float(pd.Series(s).mean())),
            high_station_rate=('is_high_station_p95', lambda s: float(pd.Series(s).mean())),
            t_pred_min=('t_pred', 'min'),
            t_pred_max=('t_pred', 'max'),
        )
    )
    return out

summary_h1 = summarize_label_index(label_h1_flagged)
summary_h24 = summarize_label_index(label_h24_flagged)

contract_summary = pd.concat([summary_h1, summary_h24], ignore_index=True)
contract_summary.head(10)


,horizon_hours,station,split,n_examples,y_missing_rate,high_global_rate,high_station_rate,t_pred_min,t_pred_max
0,1,Aotizhongxin,test,5153,0.0,0.073938,0.067728,2016-07-26 17:00:00,2017-02-28 22:00:00
1,1,Aotizhongxin,train,23813,0.0,0.055180,0.050687,2013-03-01 00:00:00,2015-12-18 14:00:00
2,1,Aotizhongxin,val,5127,0.0,0.049932,0.047201,2015-12-19 16:00:00,2016-07-24 18:00:00
3,1,Changping,test,5176,0.0,0.044629,0.057187,2016-07-25 20:00:00,2017-02-28 22:00:00
4,1,Changping,train,23902,0.0,0.039285,0.050079,2013-03-01 00:00:00,2015-12-18 11:00:00
5,1,Changping,val,5164,0.0,0.029435,0.035050,2015-12-19 16:00:00,2016-07-24 18:00:00
6,1,Dingling,test,5020,0.0,0.033267,0.047809,2016-07-25 20:00:00,2017-02-28 22:00:00
7,1,Dingling,train,24082,0.0,0.038410,0.050536,2013-03-01 00:00:00,2015-12-18 14:00:00
8,1,Dingling,val,5132,0.0,0.037997,0.045207,2015-12-19 16:00:00,2016-07-24 18:00:00
9,1,Dongsi,test,5078,0.0,0.094132,0.083104,2016-07-25 20:00:00,2017-02-28 22:00:00


## 12. Persist forecasting contract artifacts


In [15]:
split_df.to_parquet(OUT_SPLIT_DEF, index=False)
label_h1_flagged.to_parquet(OUT_LABEL_H1, index=False)
label_h24_flagged.to_parquet(OUT_LABEL_H24, index=False)
contract_summary.to_parquet(OUT_SUMMARY, index=False)

contract = {
    'contract_version': 'v1',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'inputs': {
        'assembled_path': str(ASSEMBLED_PATH),
        'station_status_policy_path': str(STATION_STATUS_PATH),
        'eda_implications_path': str(EDA_IMPLICATIONS_PATH),
    },
    'keys': KEYS,
    'target': {
        'target_col': TARGET_COL,
        'target_unit': 'as-is',
    },
    'horizons_hours': {
        'primary': H_PRIMARY,
        'secondary': H_SECONDARY,
        'all': HORIZONS_HOURS,
    },
    'availability_mode': {
        'type': 'observed_only',
        'allowed_max_timestamp': 't',
        'forbidden_future_window': '(t, t+h]',
    },
    'station_filter': {
        'eligible_statuses': sorted(list(ELIGIBLE_STATUSES)),
        'n_stations_total': int(df['station'].nunique()),
        'n_stations_eligible': int(df_filt['station'].nunique()),
    },
    'split': {
        'type': 'global_chronological',
        'grid_freq': 'H',
        'proportions': {'train': TRAIN_PROP, 'val': VAL_PROP, 'test': 1.0 - (TRAIN_PROP + VAL_PROP)},
        'purge_gap_hours': int(PURGE_GAP_HOURS),
        'assignment_rule': 'by_prediction_time_t',
        'boundary_rule': 'require_target_time_in_same_split_else_drop',
        'split_boundaries': {
            'train_start': str(base_train_start),
            'train_end': str(base_train_end),
            'train_purge_start': str(train_purge_start),
            'train_purge_end': str(train_purge_end),
            'val_start': str(val_start),
            'val_end': str(val_end),
            'val_purge_start': str(val_purge_start),
            'val_purge_end': str(val_purge_end),
            'test_start': str(test_start),
            'test_end': str(test_end),
        },
    },
    'high_pollution': {
        'definition': 'train_only_p95_of_target_y',
        'global_p95_value_h1': float(hp_meta_h1['global_p95_value']),
        'global_p95_value_h24': float(hp_meta_h24['global_p95_value']),
    },
    'artifacts': {
        'split_definition_path': str(OUT_SPLIT_DEF),
        'label_index_h1_path': str(OUT_LABEL_H1),
        'label_index_h24_path': str(OUT_LABEL_H24),
        'contract_summary_tables_path': str(OUT_SUMMARY),
    },
}

OUT_CONTRACT_JSON.write_text(json.dumps(contract, indent=2), encoding='utf-8')

# Show persisted paths for manual verification
{
    'contract_json': str(OUT_CONTRACT_JSON),
    'split_definition': str(OUT_SPLIT_DEF),
    'label_index_h1': str(OUT_LABEL_H1),
    'label_index_h24': str(OUT_LABEL_H24),
    'contract_summary': str(OUT_SUMMARY),
}


{'contract_json': 'F:\\DOCUMENTOS\\CIENCIA DE DATOS\\PROYECTOS\\portfolio_data_science\\air-quality-forecasting-environmental-signals\\data\\interim\\beijing_air_quality\\forecasting_contract\\forecasting_problem_contract.json',
 'split_definition': 'F:\\DOCUMENTOS\\CIENCIA DE DATOS\\PROYECTOS\\portfolio_data_science\\air-quality-forecasting-environmental-signals\\data\\interim\\beijing_air_quality\\forecasting_contract\\split_definition.parquet',
 'label_index_h1': 'F:\\DOCUMENTOS\\CIENCIA DE DATOS\\PROYECTOS\\portfolio_data_science\\air-quality-forecasting-environmental-signals\\data\\interim\\beijing_air_quality\\forecasting_contract\\label_index_h1.parquet',
 'label_index_h24': 'F:\\DOCUMENTOS\\CIENCIA DE DATOS\\PROYECTOS\\portfolio_data_science\\air-quality-forecasting-environmental-signals\\data\\interim\\beijing_air_quality\\forecasting_contract\\label_index_h24.parquet',
 'contract_summary': 'F:\\DOCUMENTOS\\CIENCIA DE DATOS\\PROYECTOS\\portfolio_data_science\\air-quality-forec

**Execution visibility preview (post-run):** the next cell *reads* persisted artifacts to make execution evidence visible in the notebook. It does not change any contract decision or artifact content.


In [16]:
# Artifact preview (reads only)
artifacts = [
    ('forecasting_problem_contract.json', OUT_CONTRACT_JSON),
    ('split_definition.parquet', OUT_SPLIT_DEF),
    ('label_index_h1.parquet', OUT_LABEL_H1),
    ('label_index_h24.parquet', OUT_LABEL_H24),
    ('contract_summary_tables.parquet', OUT_SUMMARY),
]

status_rows = []
for name, path in artifacts:
    status_rows.append({
        'artifact': name,
        'exists': bool(path.exists()),
        'bytes': int(path.stat().st_size) if path.exists() else None,
        'path': str(path),
    })

artifact_status = pd.DataFrame(status_rows)
artifact_status

# Split counts
split_def = pd.read_parquet(OUT_SPLIT_DEF)
split_counts = split_def['split'].value_counts(dropna=False).rename_axis('split').reset_index(name='n_hours')
split_counts

# Label counts by horizon/split (from summary)
summary = pd.read_parquet(OUT_SUMMARY)
label_counts = (
    summary.groupby(['horizon_hours', 'split'], as_index=False)['n_examples']
    .sum()
    .sort_values(['horizon_hours', 'split'])
    .reset_index(drop=True)
)
label_counts

# High-pollution thresholds (from contract JSON)
contract = json.loads(OUT_CONTRACT_JSON.read_text(encoding='utf-8'))
contract.get('high_pollution', {})


{'definition': 'train_only_p95_of_target_y',
 'global_p95_value_h1': 239.0,
 'global_p95_value_h24': 239.0}

## 13. Notebook close (handoff to baselines/modeling notebooks)

Notebook 03 ends after persisting the forecasting contract artifacts.
Next notebooks may consume the label indexes and split definition to run baselines and candidate models without redefining the contract.
